# 01 — RAG Pipeline Basics & Giao Diện Dashboard

**Vai trò:** Pipeline Engineer · **Task:** S1-PE-01 (Yêu cầu 10.1, 10.2)

Notebook này giới thiệu về cấu trúc và cách tổ chức giao diện **Streamlit Research Dashboard** (được triển khai trong task `S1-PE-01`), giải thích cơ chế quản lý trạng thái chia sẻ giữa các trang bằng `st.session_state`, và mô phỏng luồng hoạt động cơ bản của RAG Pipeline dưới góc nhìn của Pipeline Engineer.

In [1]:
import sys
from pathlib import Path

# Đảm bảo thư mục gốc dự án luôn nằm trong sys.path để import các module src và config
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import AppConfig
from src.models import DocumentType, ChunkStrategy, IndexingResult, RAGResponse

print(f"Project root: {PROJECT_ROOT}")

## 1. Cấu trúc Giao diện Đa Trang (Multi-page Dashboard)

Dashboard được tổ chức theo đúng tài liệu thiết kế (design.md §2.6) gồm **trang chủ** và **4 trang chức năng**:
- **Trang chủ (`app/main.py`)**: Giới thiệu hệ thống, sơ đồ luồng dữ liệu của RAG Pipeline, trạng thái hệ thống, và **Sidebar cấu hình** chung.
- **Document Upload (`app/pages/01_document_upload.py`)**: Cho phép người dùng tải file lên để xử lý và lưu vào cơ sở dữ liệu vector.
- **Chat Interface (`app/pages/02_chat_interface.py`)**: Giao diện hỏi đáp tương tác với LLM cục bộ.
- **Retrieval Debug (`app/pages/03_retrieval_debug.py`)**: Giúp quan sát chi tiết các chunk văn bản được truy xuất kèm theo điểm số tương đồng (similarity score).
- **Experiment Log (`app/pages/04_experiment_log.py`)**: Theo dõi các sự kiện nạp tài liệu và lịch sử truy vấn để so sánh hiệu năng các tham số cấu hình.

## 2. Quản lý Cấu hình và Trạng thái chia sẻ (`st.session_state`)

Để 4 trang hoạt động nhất quán, cấu hình của RAG Pipeline được lưu trữ tập trung tại `st.session_state["config"]` khi người dùng điều chỉnh trên sidebar ở trang chủ. Dưới đây là cách chúng ta khởi tạo cấu hình mặc định từ `AppConfig`:

In [2]:
# Khởi tạo session state giả lập trong môi trường notebook để kiểm tra
mock_session_state = {}

def init_session_defaults(state: dict) -> None:
    if "config" not in state:
        default = AppConfig()
        state["config"] = {
            "ollama_model": default.ollama.default_llm_model,
            "embedding_model": default.ollama.default_embedding_model,
            "chunk_size": default.chunker.chunk_size,
            "chunk_overlap": default.chunker.chunk_overlap,
            "top_k": default.top_k,
        }
    if "chat_history" not in state:
        state["chat_history"] = []
    if "last_retrieval_result" not in state:
        state["last_retrieval_result"] = None
    if "indexed_docs" not in state:
        state["indexed_docs"] = []

init_session_defaults(mock_session_state)
cfg = mock_session_state["config"]

print("Khởi tạo cấu hình mặc định thành công:")
print(f"  LLM Model: {cfg['ollama_model']}")
print(f"  Embedding Model: {cfg['embedding_model']}")
print(f"  Chunk Size: {cfg['chunk_size']}")
print(f"  Chunk Overlap: {cfg['chunk_overlap']}")
print(f"  Top-K: {cfg['top_k']}")

Khởi tạo cấu hình mặc định thành công:
  LLM Model: llama3
  Embedding Model: nomic-embed-text
  Chunk Size: 512
  Chunk Overlap: 50
  Top-K: 5


## 3. Mô phỏng luồng RAG Pipeline giả lập (Sprint 1)

Trong Sprint 1, để có thể demo giao diện ngay lập tức mà không cần chờ các phần lõi (như ChromaDB hay Ollama Embedding) hoàn thiện, chúng ta sử dụng cơ chế **Mock/Stub**. 

Dưới đây là cách trang **Document Upload** và **Chat Interface** tương tác với các hàm giả lập để hiển thị thông tin lên giao diện:

In [3]:
import hashlib
import time

def stub_index_document(file_name: str, config: dict) -> IndexingResult:
    """Giả lập kết quả IndexingResult phục vụ demo giao diện Sprint 1."""
    time.sleep(0.1)  # Giả lập thời gian xử lý nhỏ
    doc_id = hashlib.md5(file_name.encode()).hexdigest()[:12]
    # Mô phỏng số chunks tỉ lệ nghịch với chunk_size
    fake_num_chunks = max(1, int(1000 / config.get("chunk_size", 512)) + 2)
    return IndexingResult(
        doc_id=doc_id,
        num_chunks=fake_num_chunks,
        collection_name="rag_collection",
        success=True,
        error_message=None
    )

# Chạy thử nghiệm mô phỏng Index tài liệu
result_indexing = stub_index_document("sample_paper.pdf", cfg)
print("KẾT QUẢ MÔ PHỎNG INDEXING:")
print(f"  Tài liệu: sample_paper.pdf")
print(f"  Số chunks giả lập: {result_indexing.num_chunks}")
print(f"  Trạng thái: {'Thành công' if result_indexing.success else 'Thất bại'}")
print(f"  Doc ID: {result_indexing.doc_id}")

KẾT QUẢ MÔ PHỎNG INDEXING:
  Tài liệu: sample_paper.pdf
  Số chunks giả lập: 4
  Trạng thái: Thành công
  Doc ID: c2a8df80c85c


Tiếp theo, hãy cùng xem cách luồng hỏi đáp mô phỏng hoạt động trên trang **Chat Interface**. Nó sẽ trả về một đối tượng `RAGResponse` có cấu trúc đầy đủ chứa câu trả lời và các nguồn tham chiếu (chunks) để hiển thị chi tiết ở trang **Retrieval Debug**:

In [4]:
import datetime
from src.models import ScoredChunk, Chunk

def stub_rag_query(question: str, config: dict) -> RAGResponse:
    """Giả lập kết quả RAGResponse để hiển thị hội thoại và debug."""
    # Tạo danh sách context chunks giả lập để mô phỏng kết quả truy xuất
    fake_chunks = [
        ScoredChunk(
            chunk=Chunk(
                chunk_id=f"chunk_{i}",
                doc_id="demo_doc_001",
                content=f"[Demo Sprint 1] Đoạn văn bản mẫu số {i+1} dùng làm ngữ cảnh.",
                start_index=i * 200,
                end_index=(i + 1) * 200,
                metadata={"source": "demo_document.pdf", "page": i + 1},
            ),
            score=round(0.95 - i * 0.08, 2),
            rank=i + 1,
        )
        for i in range(min(config.get("top_k", 3), 3))
    ]

    fake_answer = (
        f"**[Sprint 1 — Demo Mode]** Đây là câu trả lời giả lập cho câu hỏi: "
        f'"{question}"\n\n'
        "Hệ thống đang chạy ở chế độ stub. Câu trả lời thật từ LLM cục bộ "
        f"({config.get('ollama_model', 'llama3')}) sẽ được kích hoạt từ Sprint 3 "
        "khi `RAGPipeline.query()` được triển khai đầy đủ (task S3-PE-03)."
    )

    return RAGResponse(
        question=question,
        answer=fake_answer,
        contexts=fake_chunks,
        model_name=config.get("ollama_model", "llama3"),
        latency_ms=1234.5,
        timestamp=datetime.datetime.now(),
    )

# Chạy truy vấn thử nghiệm
response = stub_rag_query("RAG là gì?", cfg)
print("KẾT QUẢ TRUY VẤN GIẢ LẬP:")
print(f"  Câu hỏi: {response.question}")
print(f"  Model LLM sử dụng: {response.model_name}")
print(f"  Độ trễ (latency): {response.latency_ms} ms")
print(f"\nCâu trả lời:\n{response.answer}")
print("\nCác nguồn tham chiếu đã tìm thấy:")
for sc in response.contexts:
    source_file = sc.chunk.metadata.get('source', 'unknown')
    page_num = sc.chunk.metadata.get('page', 'unknown')
    print(f"  - Rank #{sc.rank}: score={sc.score:.2f} | File: {source_file} (trang {page_num})")

KẾT QUẢ TRUY VẤN GIẢ LẬP:
  Câu hỏi: RAG là gì?
  Model LLM sử dụng: llama3
  Độ trễ (latency): 1234.5 ms

Câu trả lời:
**[Sprint 1 — Demo Mode]** Đây là câu trả lời giả lập cho câu hỏi: "RAG là gì?"

Hệ thống đang chạy ở chế độ stub. Câu trả lời thật từ LLM cục bộ (llama3) sẽ được kích hoạt từ Sprint 3 khi `RAGPipeline.query()` được triển khai đầy đủ (task S3-PE-03).

Các nguồn tham chiếu đã tìm thấy:
  - Rank #1: score=0.95 | File: demo_document.pdf (trang 1)
  - Rank #2: score=0.87 | File: demo_document.pdf (trang 2)
  - Rank #3: score=0.79 | File: demo_document.pdf (trang 3)


## 4. Tổng kết & Lộ trình Tích hợp tiếp theo

- **Đạt được ở Sprint 1**: Dựng khung giao diện Streamlit đẹp mắt, hoạt động đa trang mượt mà, đồng bộ hóa tham số cấu hình chung qua `st.session_state`, và cài đặt các cơ chế hiển thị kết quả giả lập trực quan.
- **Sprint 2 (Lắp ghép Indexing)**: Thay thế hàm `stub_index_document()` bằng luồng nạp và xử lý tài liệu thật: `DocumentLoader` → `TextChunker` → `OllamaEmbeddingModel` → `ChromaVectorStore`.
- **Sprint 3 (Lắp ghép Truy Vấn)**: Thay thế hàm `stub_rag_query()` bằng logic query thật từ `RAGPipeline.query()`, kích hoạt sinh câu trả lời từ mô hình LLM chạy cục bộ qua Ollama Client.